# Etapa 4 — Modelagem

**Objetivo:** Treinar e comparar três abordagens para classificar os 17 estados operacionais.

| Modelo | Input | Estratégia |
|--------|-------|-----------|
| **Random Forest** | 88 features estatísticas | Ensemble de árvores (bagging) |
| **XGBoost** | 88 features estatísticas | Boosting sequencial (foco em exemplos difíceis) |
| **CNN-1D (FCN)** | Série temporal bruta (300 × 8) | Rede neural convolucional sem features manuais |

RF e XGBoost usam as 88 features artesanais da Etapa 3.
A CNN-1D aprende representações diretamente da série bruta — sem nenhuma engenharia de features.
A comparação direta entre as duas abordagens é o eixo central do TCC.

**Sobre o tempo de treinamento:**
Em **modo validação** (`VALIDATION_MODE = True`), este notebook treina um RF rápido para verificar que o pipeline funciona.
Em **modo completo** (`VALIDATION_MODE = False`), carrega os modelos já treinados pelos scripts em `scripts/` (treino real: ~1–4 h cada).

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from xgboost import XGBClassifier

from config import (
    FEATURES_WINDOW_PATH,
    METRICS_DIR,
    MODELS_DIR,
    N_ITER_SEARCH,
    N_JOBS,
    N_SPLITS_CV,
    RANDOM_STATE,
    VALIDATION_MODE,
    WINDOW_CLASSES,
)

mode_label = 'VALIDACAO' if VALIDATION_MODE else 'COMPLETO'
print(f"{'='*55}")
print(f"  Modo: {mode_label}")
print(f"  Folds CV: {N_SPLITS_CV}  |  Iterações busca: {N_ITER_SEARCH}")
if VALIDATION_MODE:
    print("  RF treinado aqui (rápido). XGBoost + CNN-1D: ver scripts/.")
else:
    print("  Modelos carregados de results/models/ (pré-treinados).")
print(f"{'='*55}")

## 4.1 Preparar dados para modelagem

Carregamos `features_window_class.parquet` com as 88 features estatísticas e a coluna `window_label` — a moda dos estados operacionais em cada janela de 300 s.
NaN remanescentes (janelas de sensores inativos) são preenchidos com a mediana via `SimpleImputer`.

In [ ]:
df = pd.read_parquet(FEATURES_WINDOW_PATH)

META_COLS = ['instance_id', 'fault_class', 'window_label', 'source_type', 'window_start']
feature_cols = [c for c in df.columns if c not in META_COLS]

X_raw  = df[feature_cols].values
y      = df['window_label'].values
groups = df['instance_id'].values

imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X_raw)

classes = sorted(np.unique(y))
print(f'Janelas: {len(y):,} | Features: {X.shape[1]} | Classes: {len(classes)}')
print(f'Instâncias únicas: {len(np.unique(groups))}')
print(f'\nJanelas por estado (desbalanceamento):')
for c in classes:
    n = (y == c).sum()
    pct = 100 * n / len(y)
    label = WINDOW_CLASSES.get(c, str(c))
    print(f'  {c:>3} — {label:<30}: {n:>7,} ({pct:.1f}%)')

## 4.2 Separação treino/teste — GroupKFold

Com divisão aleatória, janelas do **mesmo poço** aparecem em treino e teste simultaneamente.
O modelo memoriza padrões do poço (faixa absoluta de pressão, operação histórica) e parece ótimo —
mas falha em poços novos. Isso é **vazamento de dados** (*data leakage*).

`GroupKFold(n_splits=5)` garante que cada poço aparece em **apenas 1 fold de teste**.
O modelo aprende de outros poços com o mesmo tipo de evento e é testado em poços desconhecidos.

`class_weight='balanced'` nas árvores: cada classe recebe um peso inversamente proporcional
à sua frequência — a classe 7 (0,2% do dataset) tem peso ~500× maior que a classe 0.

In [ ]:
gkf = GroupKFold(n_splits=N_SPLITS_CV)

print(f'GroupKFold com {N_SPLITS_CV} folds — distribuição de instâncias por fold:')
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    train_iids    = np.unique(groups[train_idx])
    test_iids     = np.unique(groups[test_idx])
    n_cls_test    = len(np.unique(y[test_idx]))
    n_cls_train   = len(np.unique(y[train_idx]))
    print(f'  Fold {fold+1}: treino={len(train_iids):>4} inst. ({n_cls_train} classes) | '
          f'teste={len(test_iids):>3} inst. ({n_cls_test} classes)')

## 4.3 Random Forest e XGBoost — features artesanais

Ambos recebem as **88 features estatísticas** e são treinados com `RandomizedSearchCV` dentro do `GroupKFold`.

**Por que XGBoost supera RF na classe 7 (PCK Incrustação)?**
O RF usa *bagging*: cada árvore vê uma amostra aleatória dos dados.
Com apenas ~180 janelas da classe 7 por fold, a maioria das árvores nunca vê esse evento.
O XGBoost usa *boosting sequencial*: cada árvore nova foca nos exemplos que as anteriores erraram.
Com esse mecanismo, a classe 7 recebe atenção progressivamente maior ao longo das 500 iterações.

**XGBoost com 3 filtros:** Gaussiano (σ=2), sem filtro e filtro estatístico adaptativo.
Ver scripts `train_xgboost_window_class.py`, `train_xgboost_nofilter.py`, `train_xgboost_statistical.py`.

In [ ]:
if VALIDATION_MODE:
    print("VALIDACAO: treinando RF rápido para verificar o pipeline...")
    print("(Resultado não representa o modelo final — use modo COMPLETO para isso)\n")

    n_splits_val = min(N_SPLITS_CV, len(np.unique(groups)))
    gkf_val = GroupKFold(n_splits=n_splits_val)

    rf_grid = {
        'n_estimators': [50, 100],
        'max_depth': [None, 10],
        'max_features': ['sqrt'],
        'class_weight': ['balanced'],
    }
    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        rf_grid, n_iter=3, cv=gkf_val,
        scoring='f1_macro', n_jobs=N_JOBS, random_state=RANDOM_STATE, verbose=1,
    )
    search.fit(X, y, groups=groups)
    print(f'\nF1-macro (CV, validação rápida): {search.best_score_:.4f}')
    print(f'Melhores parâmetros: {search.best_params_}')

else:
    print("COMPLETO: carregando modelos pré-treinados...")
    rf  = joblib.load(MODELS_DIR / 'rf_window_class.joblib')
    xgb = joblib.load(MODELS_DIR / 'xgboost_window_class.joblib')

    with open(METRICS_DIR / 'rf_window_class_metrics.json') as f:
        rf_m = json.load(f)
    with open(METRICS_DIR / 'xgboost_window_class_metrics.json') as f:
        xgb_m = json.load(f)

    rf_f1  = rf_m['metrics_concat_folds']['f1_macro']
    xgb_f1 = xgb_m['metrics_concat_folds']['f1_macro']

    print(f'\nRF (hiperparâmetros): {rf_m["best_params"]}')
    print(f'XGBoost (hiperparâmetros): {xgb_m["best_params"]}')
    print(f'\nResultados finais (GroupKFold 5 folds OOF):')
    print(f'  RF      → F1-macro: {rf_f1:.4f} | Acc: {rf_m["metrics_concat_folds"]["accuracy"]:.4f}')
    print(f'  XGBoost → F1-macro: {xgb_f1:.4f} | Acc: {xgb_m["metrics_concat_folds"]["accuracy"]:.4f}')
    print(f'\nXGBoost supera RF em {(xgb_f1 - rf_f1)*100:.1f} p.p. de F1-macro')
    print('→ Vantagem concentrada nas classes raras (ver Etapa 5 para análise por classe)')

## 4.4 CNN-1D (FCN) — Rede Neural sobre Série Temporal Bruta

Enquanto RF e XGBoost recebem as 88 features artesanais, a CNN-1D recebe a **série temporal bruta**:
300 timesteps × 8 sensores = janela de forma (300, 8).

**Arquitetura FCN (Fully Convolutional Network):**

```
Entrada: (300, 8)
↓ Conv1D(128 filtros, kernel=8) → BatchNorm → ReLU   # padrões de curto prazo
↓ Conv1D(256 filtros, kernel=5) → BatchNorm → ReLU   # combina padrões intermediários
↓ Conv1D(128 filtros, kernel=3) → BatchNorm → ReLU   # refina representação local
↓ GlobalAveragePooling1D()                             # comprime dimensão temporal
↓ Dense(17, softmax)                                   # 17 estados operacionais
```

**Detalhes de treinamento:**
- `EarlyStopping` monitora `val_f1_macro` (não `val_loss`) — evita parar antes das classes raras aprenderem
- Inferência em chunks de 8.192 janelas no callback — evita OOM durante validação

**Para treinar:** `python scripts/train_cnn1d.py` (~2–4 h com GPU)

In [ ]:
with open(METRICS_DIR / 'cnn1d_metrics.json') as f:
    cnn_m = json.load(f)

print('CNN-1D (FCN) — Resultados finais (5 folds OOF):')
print(f'  F1-macro    : {cnn_m["f1_macro"]:.4f}')
print(f'  F1-weighted : {cnn_m["f1_weighted"]:.4f}')
print(f'  Accuracy    : {cnn_m["accuracy"]:.4f}')
print(f'  F1 por fold : {[round(f, 4) for f in cnn_m["f1_per_fold"]]}')

if not VALIDATION_MODE:
    xgb_f1  = xgb_m['metrics_concat_folds']['f1_macro']
    cnn_f1  = cnn_m['f1_macro']
    delta   = xgb_f1 - cnn_f1
    print(f'\nDiferença XGBoost − CNN-1D: {delta*100:.1f} p.p. de F1-macro')

print('\nMelhores classes da CNN-1D (próximas do XGBoost):')
best_cnn = sorted(cnn_m['per_class_f1'].items(), key=lambda x: x[1], reverse=True)[:5]
for name, f1 in best_cnn:
    print(f'  {name:<30}: {f1:.3f}')

print('\nPiores classes da CNN-1D (maior déficit vs XGBoost):')
worst_cnn = sorted(cnn_m['per_class_f1'].items(), key=lambda x: x[1])[:5]
for name, f1 in worst_cnn:
    print(f'  {name:<30}: {f1:.3f}')